# Filter Images by Article IDs

This notebook filters product images from the `images_256_256` folder based on article IDs defined in `articles_filtered.csv`

**What this notebook does:**
1. Loads the filtered articles CSV file (containing only shoes)
2. Finds corresponding images in the raw images folder
3. Copies matching images to a new `images_filtered` folder
4. Reports any missing images

**Prerequisites:**
- `raw_data/articles_filtered.csv`
- `raw_data/images_256_256/`

## Step 1: Import Required Libraries

We use:
- `pandas` for reading CSV files
- `os` and `shutil` for file operations (checking paths, copying files)
- `pathlib` for path manipulation

In [17]:
import pandas as pd
import os
import shutil
from pathlib import Path

## Step 2: Load the Filtered Articles Data

Load the CSV file containing only the articles we want (shoes). This file was created by `preprocess_data.ipynb` which filtered the original `articles.csv` by product group.

In [18]:
df = pd.read_csv('../raw_data/articles_filtered.csv')
print(f"Total articles: {len(df)}")
df.head()

Total articles: 5283


,Unnamed: 0,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,124,181160009,181160,Eva chelsea boot,87,Boots,Shoes,1010016,Solid,17,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Chelsea boots with elasticated gores in the si...
1,257,212042036,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,1,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...
2,258,212042043,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,9,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...
3,259,212042066,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,22,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...
4,260,212042070,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,10,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...


## Step 3: Extract Unique Article IDs

Get all unique article IDs from the dataframe. Each article ID corresponds to one product image.

In [19]:
article_ids = df['article_id'].unique()
print(f"Unique article IDs: {len(article_ids)}")

Unique article IDs: 5283


## Step 4: Define Image Path Helper Function

Images are stored in subfolders based on the first 3 digits of the article ID.

**Example:**
- Article ID: `181160009`
- Padded to 10 digits: `0181160009`
- Subfolder: `018` (first 3 digits)
- Full path: `raw_data/images_256_256/018/0181160009.jpg`

In [ ]:
def get_image_path(article_id, base_path='../raw_data/images_256_256'):
    """
    Convert article_id to image path.

    Args:
        article_id: The product article ID (e.g., 181160009)
        base_path: Base directory containing image subfolders

    Returns:
        Full path to the image file
    """
    # Pad article_id to 10 digits with leading zeros
    article_str = str(article_id).zfill(10)
    # First 3 digits determine the subfolder
    subfolder = article_str[:3]
    # Construct filename
    filename = f"{article_str}.jpg"
    return os.path.join(base_path, subfolder, filename)

## Step 5: Test the Path Function

Verify that our path construction works correctly by testing with the first article ID.

In [21]:
test_id = article_ids[0]
test_path = get_image_path(test_id)
print(f"Article ID: {test_id}")
print(f"Image path: {test_path}")
print(f"File exists: {os.path.exists(test_path)}")

Article ID: 181160009
Image path: ../raw_data/images_256_256/018/0181160009.jpg
File exists: True


## Step 6: Check Image Availability

Iterate through all article IDs and check which images exist in the source folder. This helps us understand the data completeness before copying.

In [22]:
existing_images = []
missing_images = []

for article_id in article_ids:
    path = get_image_path(article_id)
    if os.path.exists(path):
        existing_images.append((article_id, path))
    else:
        missing_images.append((article_id, path))

print(f"Images found: {len(existing_images)}")
print(f"Images missing: {len(missing_images)}")
print(f"Coverage: {len(existing_images) / len(article_ids) * 100:.1f}%")

Images found: 5156
Images missing: 127
Coverage: 97.6%


## Step 7: Inspect Missing Images (Optional)

If there are missing images, let's see which articles don't have corresponding images. This information can be useful for data quality analysis.

In [ ]:
if missing_images:
    print(f"Article IDs without images ({len(missing_images)} total):\n")
    missing_ids = [article_id for article_id, path in missing_images]

    # Show first 20 missing IDs with expected paths
    for article_id in missing_ids[:20]:
        print(f"  - {article_id} (expected: {get_image_path(article_id)})")
    if len(missing_ids) > 20:
        print(f"  ... and {len(missing_ids) - 20} more")

    # Show details of missing articles
    df_missing = df[df['article_id'].isin(missing_ids)][['article_id', 'prod_name', 'product_type_name']]
    print(f"\nMissing articles details:")
    display(df_missing.head(10))
else:
    print("All images found!")

Article IDs without images (127 total):

  - 212042043 (expected: ../raw_data/images_256_256/021/0212042043.jpg)
  - 212042066 (expected: ../raw_data/images_256_256/021/0212042066.jpg)
  - 438702006 (expected: ../raw_data/images_256_256/043/0438702006.jpg)
  - 460363012 (expected: ../raw_data/images_256_256/046/0460363012.jpg)
  - 461414009 (expected: ../raw_data/images_256_256/046/0461414009.jpg)
  - 461414010 (expected: ../raw_data/images_256_256/046/0461414010.jpg)
  - 469039021 (expected: ../raw_data/images_256_256/046/0469039021.jpg)
  - 470985008 (expected: ../raw_data/images_256_256/047/0470985008.jpg)
  - 470985010 (expected: ../raw_data/images_256_256/047/0470985010.jpg)
  - 475791011 (expected: ../raw_data/images_256_256/047/0475791011.jpg)
  - 475827007 (expected: ../raw_data/images_256_256/047/0475827007.jpg)
  - 510415003 (expected: ../raw_data/images_256_256/051/0510415003.jpg)
  - 510504001 (expected: ../raw_data/images_256_256/051/0510504001.jpg)
  - 515074002 (expected

,article_id,prod_name,product_type_name
2,212042043,Mimmi sneaker,Sneakers
3,212042066,Mimmi sneaker,Sneakers
51,438702006,LENA SNEAKER,Sneakers
69,460363012,Sivert sneaker,Sneakers
73,461414009,OL LINUS PQ loafer,Flat shoe
74,461414010,OL LINUS PQ loafer,Flat shoe
78,469039021,Henrik ballerina,Ballerinas
88,470985008,Vaughan sneaker,Sneakers
90,470985010,Vaughan sneaker,Sneakers
98,475791011,Pontus Espadrill,Sandals


## Step 8: Create Output Directory

Create the destination folder for filtered images. If it already exists, no error will be raised.

In [24]:
output_dir = '../raw_data/images_filtered'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

Output directory: ../raw_data/images_filtered


## Step 9: Copy Filtered Images

Copy all existing images to the output directory. Progress is printed every 500 images.

**Note:** This cell may take a while depending on the number of images.

In [25]:
copied = 0
for article_id, src_path in existing_images:
    filename = os.path.basename(src_path)
    dst_path = os.path.join(output_dir, filename)
    shutil.copy2(src_path, dst_path)
    copied += 1
    if copied % 500 == 0:
        print(f"Copied {copied}/{len(existing_images)} images...")

print(f"\nDone! Copied {copied} images to {output_dir}")

Copied 500/5156 images...
Copied 1000/5156 images...
Copied 1500/5156 images...
Copied 2000/5156 images...
Copied 2500/5156 images...
Copied 3000/5156 images...
Copied 3500/5156 images...
Copied 4000/5156 images...
Copied 4500/5156 images...
Copied 5000/5156 images...

Done! Copied 5156 images to ../raw_data/images_filtered


## Step 10: Verify Results

Confirm that all images were copied successfully by counting files in the output directory.

In [26]:
filtered_images = os.listdir(output_dir)
# Filter out hidden files like .DS_Store
filtered_images = [f for f in filtered_images if not f.startswith('.')]
print(f"Images in output folder: {len(filtered_images)}")
print(f"Expected: {len(existing_images)}")
print(f"Match: {'Yes' if len(filtered_images) == len(existing_images) else 'No'}")

Images in output folder: 5156
Expected: 5156
Match: Yes
